# LiDAR Engine v0.8-alpha5.1 BeamEvidenceLite Upload Harness

Upload two files when prompted:

1. `lidar_lenses_wave_v080_alpha5_1.py`
2. `lidar_wave_test_plan_v080_alpha5_1.xlsx`

The notebook runs every enabled row in the `sweep` sheet and downloads `lidar_wave_test_outputs_v080_alpha5_1.zip`.

In [ ]:
from pathlib import Path
import glob, os, sys

try:
    from google.colab import files
    print("Upload the engine .py and the v0.8-alpha5.1 test-plan .xlsx.")
    uploaded = files.upload()
except Exception:
    uploaded = {}
    print("Not running in Colab upload mode. Looking for files in the current folder.")

def find_first(patterns, label):
    matches = []
    for pat in patterns:
        matches.extend(glob.glob(pat))
    matches = sorted(set(matches))
    if not matches:
        raise FileNotFoundError(f"Could not find {label}. Tried: {patterns}")
    return matches[-1]

ENGINE_PATH = find_first(["*v080_alpha5_1*.py", "lidar_lenses_wave*.py"], "engine .py")
PLAN_PATH = find_first(["*v080_alpha5_1*.xlsx", "lidar_wave_test_plan*.xlsx"], "test plan .xlsx")
print("ENGINE_PATH =", ENGINE_PATH)
print("PLAN_PATH   =", PLAN_PATH)

In [ ]:
from pathlib import Path
HARNESS_CODE = '\n"""\nv080_alpha5_1_upload_harness.py\n\nSpreadsheet-driven Colab/local harness for LiDAR Engine v0.8.0-alpha5.1 BeamEvidenceLite.\n\nFlow modeled after the older v0.6.10 harness:\n  1. Upload/select engine .py\n  2. Upload/select test plan .xlsx\n  3. Run every enabled row in sheet "sweep"\n  4. Save per-run contact sheets, channels.npz, masks/material views, diagnostics.json\n  5. Save summary CSV/JSON reports\n  6. Zip the whole output folder for download\n\nExpected uploads in Colab:\n  - lidar_lenses_wave_v080_alpha5_1.py\n  - lidar_wave_test_plan_v080_alpha5_1.xlsx\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport ast\nimport csv\nimport glob\nimport importlib.util\nimport json\nimport math\nimport os\nimport re\nimport shutil\nimport sys\nimport zipfile\nfrom pathlib import Path\nfrom typing import Any, Dict, Iterable, List, Optional, Tuple\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageDraw\n\n\nDEFAULT_OUT = "v080_alpha5_1_test_outputs"\n\n\ndef find_first(patterns: Iterable[str], label: str) -> str:\n    matches: List[str] = []\n    for pat in patterns:\n        matches.extend(glob.glob(pat))\n    matches = sorted(set(matches))\n    if not matches:\n        raise FileNotFoundError(f"Could not find {label}. Tried: {list(patterns)}")\n    return matches[-1]\n\n\ndef load_engine(engine_path: str):\n    engine_path = str(engine_path)\n    spec = importlib.util.spec_from_file_location("llw_v080_alpha5_1", engine_path)\n    if spec is None or spec.loader is None:\n        raise RuntimeError(f"Could not import engine from {engine_path!r}")\n    llw = importlib.util.module_from_spec(spec)\n    sys.modules["llw_v080_alpha5_1"] = llw\n    spec.loader.exec_module(llw)\n    return llw\n\n\ndef _safe_name(value: Any) -> str:\n    s = str(value).strip()\n    s = re.sub(r"[^A-Za-z0-9_.-]+", "_", s)\n    return s.strip("_") or "scene"\n\n\ndef _to_bool(value: Any, default: bool = False) -> bool:\n    if value is None:\n        return default\n    try:\n        if pd.isna(value):\n            return default\n    except Exception:\n        pass\n    if isinstance(value, bool):\n        return value\n    if isinstance(value, (int, float)):\n        return bool(value)\n    s = str(value).strip().lower()\n    if s in ("1", "true", "t", "yes", "y", "on"):\n        return True\n    if s in ("0", "false", "f", "no", "n", "off"):\n        return False\n    return default\n\n\ndef _to_int(value: Any, default: Optional[int] = None) -> Optional[int]:\n    try:\n        if value is None or pd.isna(value):\n            return default\n        return int(value)\n    except Exception:\n        return default\n\n\ndef _to_float(value: Any, default: Optional[float] = None) -> Optional[float]:\n    try:\n        if value is None or pd.isna(value):\n            return default\n        return float(value)\n    except Exception:\n        return default\n\n\ndef _to_str(value: Any, default: Optional[str] = None) -> Optional[str]:\n    try:\n        if value is None or pd.isna(value):\n            return default\n    except Exception:\n        pass\n    s = str(value).strip()\n    return s if s else default\n\n\ndef _rot(llw, rx=0, ry=0, rz=0):\n    return llw.make_rotation_matrix(rx, ry, rz)\n\n\ndef _prim(llw, shape, center, half, color, piece_id, piece_type, rot=None, transparency=0.0):\n    if rot is None:\n        rot = _rot(llw)\n    return llw.Primitive(\n        shape=shape,\n        center=np.array(center, float),\n        half_extents=np.array(half, float),\n        rotation_matrix=rot,\n        inv_rotation_matrix=rot.T,\n        color=tuple(float(c) for c in color),\n        piece_id=int(piece_id),\n        piece_type=str(piece_type),\n        transparency=float(transparency),\n    )\n\n\ndef build_occluder_gate(llw):\n    """Small v0.6.10-style gate + foliage + background wall stress scene."""\n    prims = []\n    pid = 1\n    prims.append(_prim(llw, "box", [0, -0.5, 0], [7, 0.5, 7], [0.55, 0.50, 0.38], pid, "ground")); pid += 1\n    for x in [-2.2, -1.4, -0.6, 0.6, 1.4, 2.2]:\n        prims.append(_prim(llw, "cylinder", [x, 0.7, 1.2], [0.06, 0.7, 0.06], [0.82, 0.76, 0.60], pid, "wood")); pid += 1\n    prims.append(_prim(llw, "box", [0, 1.1, 1.2], [2.7, 0.08, 0.06], [0.82, 0.76, 0.60], pid, "wood")); pid += 1\n    prims.append(_prim(llw, "sphere", [0, 1.35, 1.25], [1.2, 0.75, 0.4], [0.25, 0.70, 0.25], pid, "foliage", transparency=0.55)); pid += 1\n    prims.append(_prim(llw, "box", [0, 0.7, 2.8], [2.2, 0.7, 0.08], [0.35, 0.35, 0.35], pid, "stone")); pid += 1\n    if hasattr(llw, "apply_material_prior_transparency"):\n        prims = llw.apply_material_prior_transparency(prims)\n    return prims\n\n\ndef scene_from_name(llw, name: str):\n    n = str(name).strip().lower()\n    if n in ("demo", "cabin", "cabin_demo", "mini_saloon", "indoor", "demo_compact"):\n        prims = llw._build_demo_scene()\n    elif n in ("material_targets", "material_board", "material", "material_scan"):\n        prims = llw._build_material_target_board_scene()\n    elif n in ("occluder_gate", "occlusion", "outdoor_occlusion"):\n        return build_occluder_gate(llw)\n    else:\n        raise ValueError(f"Unknown scene {name!r}. Supported: demo, material_board, occluder_gate.")\n    if hasattr(llw, "apply_material_prior_transparency"):\n        prims = llw.apply_material_prior_transparency(prims)\n    return prims\n\n\ndef load_plan(plan_path: str) -> pd.DataFrame:\n    plan = pd.read_excel(plan_path, sheet_name="sweep")\n    if "enabled" not in plan.columns:\n        plan["enabled"] = True\n    plan = plan[plan["enabled"].map(lambda v: _to_bool(v, True))].copy()\n    if plan.empty:\n        raise ValueError("No enabled rows found in sheet \'sweep\'.")\n    return plan\n\n\ndef row_overrides(row: pd.Series) -> Dict[str, Any]:\n    int_cols = [\n        "width", "height", "rays_per_pixel", "stack", "pilot_rays",\n        "min_return_hit_count"\n    ]\n    float_cols = [\n        "edge_anti_min", "adaptive_edge_percentile", "edge_anti_max",\n        "beam_width", "min_ray_weight", "return_mix_gain",\n        "return_q_near", "return_q_far",\n        "return_expected_spread_rel", "return_expected_spread_grad",\n        "min_return_coverage",\n        "beam_split_min_weight_frac", "beam_split_min_abs_gap",\n        "beam_split_confidence_gain", "beam_coherence_mix_gain",\n        "beam_coherence_split_gain",\n        "partial_occluder_mix_min", "foliage_mix_min", "hard_partial_mix_min",\n        "beam_split_min", "beam_split_front_min", "beam_split_back_min",\n        "beam_coherence_solid_min",\n        "partial_occluder_gap_min", "partial_occluder_gap_ratio_min",\n        "partial_occluder_back_ratio_min", "partial_occluder_excess_min",\n        "partial_occluder_coherence_max", "coherent_surface_guard",\n        "fov_deg", "depth_wavelength", "attenuation_per_meter",\n        "min_coverage", "min_depth_span"\n    ]\n    str_cols = [\n        "camera", "lens", "sampling_mode", "beam_profile", "carrier_mode",\n        "edge_score_mode", "edge_fusion_mode", "classification_style",\n        "classification_priority"\n    ]\n    bool_cols = [\n        "include_ultrasonic", "include_polarization", "material_labels",\n        "auto_frame", "wave"\n    ]\n\n    overrides: Dict[str, Any] = {}\n\n    for k in int_cols:\n        if k in row:\n            v = _to_int(row.get(k), None)\n            if v is not None:\n                overrides[k] = v\n\n    for k in float_cols:\n        if k in row:\n            v = _to_float(row.get(k), None)\n            if v is not None:\n                overrides[k] = v\n\n    for k in str_cols:\n        if k in row:\n            v = _to_str(row.get(k), None)\n            if v is not None:\n                overrides[k] = v\n\n    for k in bool_cols:\n        if k in row:\n            try:\n                missing = pd.isna(row.get(k))\n            except Exception:\n                missing = False\n            if not missing:\n                overrides[k] = _to_bool(row.get(k), False)\n\n    # Optional panels column accepts a Python/JSON list or comma-separated string.\n    if "panels" in row:\n        v = _to_str(row.get("panels"), None)\n        if v:\n            try:\n                parsed = ast.literal_eval(v)\n                if isinstance(parsed, (list, tuple)):\n                    overrides["panels"] = [str(x).strip() for x in parsed]\n                else:\n                    overrides["panels"] = [x.strip() for x in v.split(",") if x.strip()]\n            except Exception:\n                overrides["panels"] = [x.strip() for x in v.split(",") if x.strip()]\n\n    return overrides\n\n\ndef pct_from_channel(channels: Dict[str, np.ndarray], hit: np.ndarray, name: str, pct: float) -> Optional[float]:\n    arr = channels.get(name)\n    if arr is None or hit is None or not np.any(hit):\n        return None\n    vals = np.asarray(arr)[hit]\n    vals = vals[np.isfinite(vals)]\n    if vals.size == 0:\n        return None\n    return float(np.percentile(vals, pct))\n\n\n\n\ndef candidate_counts_from_channels(channels: Dict[str, np.ndarray], diag: Dict[str, Any]) -> Dict[str, Any]:\n    """Alpha5 helper: count evidence candidates before exclusive class priority.\n\n    These are diagnostic counts, not final labels. They help answer whether\n    geom_edge or foliage priority is swallowing plausible partial-occluder\n    evidence.\n    """\n    if not channels:\n        return {}\n    hit = np.asarray(channels.get("hit_count", np.zeros((1, 1)))) > 0\n    hit_px = int(hit.sum()) or 1\n    ck = dict(diag.get("classifier_kwargs") or {})\n\n    def arr(name: str, default: float = 0.0):\n        base = next(iter(channels.values()))\n        return np.asarray(channels.get(name, np.zeros_like(base, dtype=float) + default), dtype=float)\n\n    ret_valid = arr("return_valid_stats") > 0.5\n    coverage = arr("return_coverage")\n    split = arr("beam_split_score")\n    front = arr("beam_front_strength")\n    back = arr("beam_back_strength")\n    gap = arr("beam_mode_gap")\n    gap_ratio = arr("beam_mode_gap_ratio")\n    if not np.any(gap_ratio):\n        depth_for_gap = arr("return_depth_mean")\n        gap_ratio = np.where(depth_for_gap > 1e-9, np.clip(gap / np.maximum(depth_for_gap, 1e-9), 0.0, 1.0), 0.0)\n    back_ratio = arr("beam_back_ratio")\n    if not np.any(back_ratio):\n        back_ratio = np.where(front > 1e-9, np.clip(back / np.maximum(front, 1e-9), 0.0, 1.0), 0.0)\n    excess = arr("return_depth_spread_excess")\n    mix = arr("return_mix_score")\n    coherence = arr("beam_coherence")\n    edge_score = arr("edge_score_geom")\n\n    min_cov = float(ck.get("min_return_coverage", diag.get("min_return_coverage", 0.25)) or 0.25)\n    min_hits = int(ck.get("min_return_hit_count", diag.get("min_return_hit_count", 4)) or 4)\n    split_min = float(ck.get("beam_split_min", diag.get("beam_split_min", 0.22)) or 0.22)\n    front_min = float(ck.get("beam_split_front_min", diag.get("beam_split_front_min", 0.10)) or 0.10)\n    back_min = float(ck.get("beam_split_back_min", diag.get("beam_split_back_min", 0.07)) or 0.07)\n    gap_min = float(ck.get("partial_occluder_gap_min", diag.get("partial_occluder_gap_min", 0.20)) or 0.20)\n    gap_ratio_min = float(ck.get("partial_occluder_gap_ratio_min", diag.get("partial_occluder_gap_ratio_min", 0.015)) or 0.015)\n    back_ratio_min = float(ck.get("partial_occluder_back_ratio_min", diag.get("partial_occluder_back_ratio_min", 0.12)) or 0.12)\n    excess_min = float(ck.get("partial_occluder_excess_min", diag.get("partial_occluder_excess_min", 0.05)) or 0.05)\n    mix_min = float(ck.get("partial_occluder_mix_min", diag.get("partial_occluder_mix_min", 0.020)) or 0.020)\n    coh_max = float(ck.get("partial_occluder_coherence_max", diag.get("partial_occluder_coherence_max", 0.82)) or 0.82)\n    solid_min = float(ck.get("beam_coherence_solid_min", diag.get("beam_coherence_solid_min", 0.45)) or 0.45)\n    coherent_guard = float(ck.get("coherent_surface_guard", diag.get("coherent_surface_guard", 0.92)) or 0.92)\n    edge_thr = float(diag.get("edge_threshold_used") or ck.get("edge_anti_min", 0.08) or 0.08)\n\n    valid = hit & ret_valid & (coverage >= min_cov) & (arr("hit_count") >= min_hits)\n    split_candidate = valid & (split > split_min)\n    front_back_candidate = valid & (front > front_min) & (back > back_min)\n    gap_candidate = valid & (gap > gap_min)\n    gap_ratio_candidate = valid & (gap_ratio > gap_ratio_min)\n    back_ratio_candidate = valid & (back_ratio > back_ratio_min)\n    excess_candidate = valid & (excess > excess_min)\n    coherent_guard_candidate = valid & (coherence > coherent_guard) & (split < max(split_min, 0.30)) & (gap_ratio < max(gap_ratio_min, 0.02))\n    partial_candidate = (\n        split_candidate & front_back_candidate & gap_candidate & gap_ratio_candidate\n        & back_ratio_candidate & excess_candidate\n        & (mix > mix_min) & (coherence < coh_max) & ~coherent_guard_candidate\n    )\n    geom_edge_candidate = hit & (edge_score > edge_thr)\n    coherence_solid_candidate = valid & (coherence > solid_min) & (split < split_min) & (mix < max(0.04, mix_min * 2.0))\n    split_or_back_candidate = valid & ((split > split_min) | (back > back_min))\n\n    out = {\n        "valid_return_candidate_count": int(valid.sum()),\n        "split_candidate_count": int(split_candidate.sum()),\n        "front_back_candidate_count": int(front_back_candidate.sum()),\n        "gap_candidate_count": int(gap_candidate.sum()),\n        "gap_ratio_candidate_count": int(gap_ratio_candidate.sum()),\n        "back_ratio_candidate_count": int(back_ratio_candidate.sum()),\n        "excess_candidate_count": int(excess_candidate.sum()),\n        "coherent_guard_candidate_count": int(coherent_guard_candidate.sum()),\n        "partial_candidate_count": int(partial_candidate.sum()),\n        "geom_edge_candidate_count": int(geom_edge_candidate.sum()),\n        "coherence_solid_candidate_count": int(coherence_solid_candidate.sum()),\n        "split_or_back_candidate_count": int(split_or_back_candidate.sum()),\n        "partial_geom_overlap_candidate_count": int((partial_candidate & geom_edge_candidate).sum()),\n    }\n    for k, v in list(out.items()):\n        out[k.replace("_count", "_frac_of_hit_px")] = float(v / hit_px)\n    return out\n\n\ndef candidate_masks_from_channels(channels: Dict[str, np.ndarray], diag: Dict[str, Any]) -> Dict[str, np.ndarray]:\n    """Return named boolean masks for alpha5.1 gate-debug contact sheets.\n\n    This mirrors candidate_counts_from_channels, but keeps every gate as a\n    separately viewable mask so the output zip shows which condition is doing\n    the work: split, front/back, gap ratio, back ratio, excess spread,\n    coherence guard, and final partial-candidate agreement.\n    """\n    if not channels:\n        return {}\n\n    base = np.asarray(next(iter(channels.values())))\n    hit = np.asarray(channels.get("hit_count", np.zeros_like(base))) > 0\n    ck = dict(diag.get("classifier_kwargs") or {})\n\n    def arr(name: str, default: float = 0.0):\n        return np.asarray(channels.get(name, np.zeros_like(base, dtype=float) + default), dtype=float)\n\n    ret_valid = arr("return_valid_stats") > 0.5\n    coverage = arr("return_coverage")\n    hit_count = arr("hit_count")\n    split = arr("beam_split_score")\n    front = arr("beam_front_strength")\n    back = arr("beam_back_strength")\n    gap = arr("beam_mode_gap")\n    depth = arr("return_depth_mean")\n    gap_ratio = arr("beam_mode_gap_ratio")\n    if not np.any(gap_ratio):\n        gap_ratio = np.where(depth > 1e-9, np.clip(gap / np.maximum(depth, 1e-9), 0.0, 1.0), 0.0)\n    back_ratio = arr("beam_back_ratio")\n    if not np.any(back_ratio):\n        back_ratio = np.where(front > 1e-9, np.clip(back / np.maximum(front, 1e-9), 0.0, 1.0), 0.0)\n    excess = arr("return_depth_spread_excess")\n    mix = arr("return_mix_score")\n    coherence = arr("beam_coherence")\n    edge_score = arr("edge_score_geom")\n\n    min_cov = float(ck.get("min_return_coverage", diag.get("min_return_coverage", 0.25)) or 0.25)\n    min_hits = int(ck.get("min_return_hit_count", diag.get("min_return_hit_count", 4)) or 4)\n    split_min = float(ck.get("beam_split_min", diag.get("beam_split_min", 0.22)) or 0.22)\n    front_min = float(ck.get("beam_split_front_min", diag.get("beam_split_front_min", 0.10)) or 0.10)\n    back_min = float(ck.get("beam_split_back_min", diag.get("beam_split_back_min", 0.07)) or 0.07)\n    gap_min = float(ck.get("partial_occluder_gap_min", diag.get("partial_occluder_gap_min", 0.20)) or 0.20)\n    gap_ratio_min = float(ck.get("partial_occluder_gap_ratio_min", diag.get("partial_occluder_gap_ratio_min", 0.015)) or 0.015)\n    back_ratio_min = float(ck.get("partial_occluder_back_ratio_min", diag.get("partial_occluder_back_ratio_min", 0.12)) or 0.12)\n    excess_min = float(ck.get("partial_occluder_excess_min", diag.get("partial_occluder_excess_min", 0.05)) or 0.05)\n    mix_min = float(ck.get("partial_occluder_mix_min", diag.get("partial_occluder_mix_min", 0.020)) or 0.020)\n    coh_max = float(ck.get("partial_occluder_coherence_max", diag.get("partial_occluder_coherence_max", 0.82)) or 0.82)\n    solid_min = float(ck.get("beam_coherence_solid_min", diag.get("beam_coherence_solid_min", 0.45)) or 0.45)\n    coherent_guard = float(ck.get("coherent_surface_guard", diag.get("coherent_surface_guard", 0.92)) or 0.92)\n    edge_thr = float(diag.get("edge_threshold_used") or ck.get("edge_anti_min", 0.08) or 0.08)\n\n    valid = hit & ret_valid & (coverage >= min_cov) & (hit_count >= min_hits)\n    split_candidate = valid & (split > split_min)\n    front_back_candidate = valid & (front > front_min) & (back > back_min)\n    gap_candidate = valid & (gap > gap_min)\n    gap_ratio_candidate = valid & (gap_ratio > gap_ratio_min)\n    back_ratio_candidate = valid & (back_ratio > back_ratio_min)\n    excess_candidate = valid & (excess > excess_min)\n    mix_candidate = valid & (mix > mix_min)\n    coherence_candidate = valid & (coherence < coh_max)\n    coherent_guard_candidate = valid & (coherence > coherent_guard) & (split < max(split_min, 0.30)) & (gap_ratio < max(gap_ratio_min, 0.02))\n    partial_candidate = (\n        split_candidate & front_back_candidate & gap_candidate & gap_ratio_candidate\n        & back_ratio_candidate & excess_candidate & mix_candidate\n        & coherence_candidate & ~coherent_guard_candidate\n    )\n    geom_edge_candidate = hit & (edge_score > edge_thr)\n    coherence_solid_candidate = valid & (coherence > solid_min) & (split < split_min) & (mix < max(0.04, mix_min * 2.0))\n    split_or_back_candidate = valid & ((split > split_min) | (back > back_min))\n\n    return {\n        "hit": hit,\n        "valid_return": valid,\n        "split_candidate": split_candidate,\n        "front_back_candidate": front_back_candidate,\n        "gap_candidate": gap_candidate,\n        "gap_ratio_candidate": gap_ratio_candidate,\n        "back_ratio_candidate": back_ratio_candidate,\n        "excess_candidate": excess_candidate,\n        "mix_candidate": mix_candidate,\n        "coherence_candidate": coherence_candidate,\n        "coherent_guard_candidate": coherent_guard_candidate,\n        "partial_candidate": partial_candidate,\n        "geom_edge_candidate": geom_edge_candidate,\n        "coherence_solid_candidate": coherence_solid_candidate,\n        "split_or_back_candidate": split_or_back_candidate,\n        "partial_geom_overlap_candidate": partial_candidate & geom_edge_candidate,\n    }\n\n\ndef _mask_panel(mask: np.ndarray, title: str, size: Tuple[int, int] = (180, 124)) -> Image.Image:\n    """Render one boolean mask panel for gate-debug sheets."""\n    m = np.asarray(mask, dtype=bool)\n    if m.ndim != 2:\n        m = np.zeros((size[1], size[0]), dtype=bool)\n    img = Image.fromarray((m.astype(np.uint8) * 255), mode="L").convert("RGB")\n    img = img.resize(size, Image.Resampling.NEAREST)\n    panel = Image.new("RGB", (size[0], size[1] + 22), (245, 245, 245))\n    panel.paste(img, (0, 22))\n    draw = ImageDraw.Draw(panel)\n    draw.rectangle([0, 0, size[0] - 1, 21], fill=(28, 32, 42))\n    draw.text((5, 5), title[:32], fill=(255, 255, 255))\n    return panel\n\n\ndef save_candidate_gate_sheet(channels: Dict[str, np.ndarray], diag: Dict[str, Any], out_path: Path) -> Optional[str]:\n    """Save a compact visual sheet of alpha5.1 gate masks for one run."""\n    masks = candidate_masks_from_channels(channels, diag)\n    if not masks:\n        return None\n    ordered = [\n        "hit", "valid_return", "split_candidate", "front_back_candidate",\n        "gap_candidate", "gap_ratio_candidate", "back_ratio_candidate", "excess_candidate",\n        "mix_candidate", "coherence_candidate", "coherent_guard_candidate", "partial_candidate",\n        "geom_edge_candidate", "coherence_solid_candidate", "split_or_back_candidate", "partial_geom_overlap_candidate",\n    ]\n    panels = [_mask_panel(masks[k], k) for k in ordered if k in masks]\n    if not panels:\n        return None\n    cols = 4\n    w, h = panels[0].size\n    rows = int(math.ceil(len(panels) / cols))\n    sheet = Image.new("RGB", (cols * w, rows * h), (230, 230, 230))\n    for i, panel in enumerate(panels):\n        sheet.paste(panel, ((i % cols) * w, (i // cols) * h))\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n    sheet.save(out_path)\n    return str(out_path)\n\n\ndef gate_breakdown_rows(beam_rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:\n    """Long-form gate counts/fracs for easier plotting and spreadsheet review."""\n    gates = [\n        "valid_return_candidate", "split_candidate", "front_back_candidate",\n        "gap_candidate", "gap_ratio_candidate", "back_ratio_candidate",\n        "excess_candidate", "coherent_guard_candidate", "partial_candidate",\n        "geom_edge_candidate", "coherence_solid_candidate", "split_or_back_candidate",\n        "partial_geom_overlap_candidate",\n    ]\n    rows: List[Dict[str, Any]] = []\n    for r in beam_rows:\n        for gate in gates:\n            rows.append({\n                "run_id": r.get("run_id"),\n                "scene": r.get("scene"),\n                "preset": r.get("preset"),\n                "beam_width": r.get("beam_width"),\n                "gate": gate,\n                "count": r.get(f"{gate}_count"),\n                "frac_of_hit_px": r.get(f"{gate}_frac_of_hit_px"),\n                "hit_pixels": r.get("hit_pixels"),\n                "partial_occluder_frac_of_hit_px": r.get("partial_occluder_frac_of_hit_px"),\n                "solid_surface_frac_of_hit_px": r.get("solid_surface_frac_of_hit_px"),\n                "uncertain_frac_of_hit_px": r.get("uncertain_frac_of_hit_px"),\n            })\n    return rows\n\n\ndef class_balance_rows(beam_rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:\n    classes = ["geom_edge", "partial_occluder", "foliage", "solid_surface", "uncertain"]\n    rows: List[Dict[str, Any]] = []\n    for r in beam_rows:\n        for cls in classes:\n            rows.append({\n                "run_id": r.get("run_id"),\n                "scene": r.get("scene"),\n                "preset": r.get("preset"),\n                "class": cls,\n                "count": r.get(f"{cls}_count"),\n                "frac_of_hit_px": r.get(f"{cls}_frac_of_hit_px"),\n                "hit_pixels": r.get("hit_pixels"),\n            })\n    return rows\n\n\ndef edge_summary_rows_from_beam_rows(beam_rows: List[Dict[str, Any]], all_diag: List[Dict[str, Any]]) -> List[Dict[str, Any]]:\n    """Alpha5.1: build edge/gate summary from channel summaries, not only diagnostics.\n\n    Earlier alpha5 diagnostics omitted some derived ratio percentiles from the\n    compact edge summary even though the npz had those channels. This uses\n    beam_rows so beam_back_ratio_p95 and beam_mode_gap_ratio_p95 are visible.\n    """\n    diag_by_run = {d.get("run_id"): d for d in all_diag}\n    cols = [\n        "run_id", "scene", "preset", "coverage", "hit_pixels",\n        "edge_threshold_used", "edge_fusion_mode", "beam_profile", "beam_width",\n        "geom_edge_frac_of_hit_px", "partial_occluder_frac_of_hit_px",\n        "foliage_frac_of_hit_px", "solid_surface_frac_of_hit_px", "uncertain_frac_of_hit_px",\n        "edge_score_geom_p95", "edge_score_raw_p95", "return_mix_score_p95",\n        "return_depth_spread_excess_p95", "beam_split_score_p95",\n        "beam_mode_gap_p95", "beam_mode_gap_ratio_p95", "beam_back_ratio_p95",\n        "beam_coherence_p50", "beam_coherence_p95",\n        "partial_candidate_count", "partial_candidate_frac_of_hit_px",\n        "gap_ratio_candidate_count", "gap_ratio_candidate_frac_of_hit_px",\n        "back_ratio_candidate_count", "back_ratio_candidate_frac_of_hit_px",\n        "coherent_guard_candidate_count", "coherent_guard_candidate_frac_of_hit_px",\n        "partial_geom_overlap_candidate_count", "partial_geom_overlap_candidate_frac_of_hit_px",\n        "coherence_solid_candidate_count", "coherence_solid_candidate_frac_of_hit_px",\n        "return_coverage_p50",\n    ]\n    rows = []\n    for r in beam_rows:\n        row = {k: r.get(k) for k in cols if k in r}\n        d = diag_by_run.get(r.get("run_id"), {})\n        for k in ["edge_score_mode", "edge_anti_min", "edge_anti_max", "adaptive_edge_percentile"]:\n            if k in d and k not in row:\n                row[k] = d.get(k)\n        rows.append(row)\n    return rows\n\ndef channel_summary(result: Dict[str, Any], run_id: int, scene: str, preset: str) -> Dict[str, Any]:\n    ch = result.get("channels", {})\n    diag = result.get("diagnostics", {})\n    hit = np.asarray(ch.get("hit_count", np.zeros((1, 1)))) > 0\n    total_px = int(hit.size)\n    hit_px = int(hit.sum())\n    labels = diag.get("classification_counts") or {}\n    row = {\n        "run_id": run_id,\n        "scene": scene,\n        "preset": preset,\n        "coverage": float((diag.get("depth_stats") or {}).get("coverage", 0.0)),\n        "hit_pixels": hit_px,\n        "total_pixels": total_px,\n        "geom_edge_count": int(labels.get("geom_edge", 0)),\n        "partial_occluder_count": int(labels.get("partial_occluder", 0)),\n        "foliage_count": int(labels.get("foliage", 0)),\n        "solid_surface_count": int(labels.get("solid_surface", 0)),\n        "uncertain_count": int(labels.get("uncertain", 0)),\n        "geom_edge_frac_of_hit_px": float(int(labels.get("geom_edge", 0)) / max(hit_px, 1)),\n        "partial_occluder_frac_of_hit_px": float(int(labels.get("partial_occluder", 0)) / max(hit_px, 1)),\n        "foliage_frac_of_hit_px": float(int(labels.get("foliage", 0)) / max(hit_px, 1)),\n        "solid_surface_frac_of_hit_px": float(int(labels.get("solid_surface", 0)) / max(hit_px, 1)),\n        "uncertain_frac_of_hit_px": float(int(labels.get("uncertain", 0)) / max(hit_px, 1)),\n        "beam_profile": diag.get("beam_profile"),\n        "beam_width": diag.get("beam_width"),\n        "edge_fusion_mode": diag.get("edge_fusion_mode"),\n        "edge_threshold_used": diag.get("edge_threshold_used"),\n    }\n    row.update(candidate_counts_from_channels(ch, diag))\n    for name in [\n        "return_mix_score_raw", "return_mix_score",\n        "return_depth_spread", "return_depth_spread_excess",\n        "return_coverage", "return_valid_stats", "return_weight_sum",\n        "return_expected_spread", "return_mix_support",\n        "beam_split_score", "beam_split_support", "beam_front_depth",\n        "beam_back_depth", "beam_front_strength", "beam_back_strength",\n        "beam_back_ratio", "beam_mode_gap", "beam_mode_gap_ratio",\n        "beam_mode_confidence", "beam_coherence",\n        "edge_score_geom", "edge_score_raw"\n    ]:\n        for p in [50, 90, 95, 99]:\n            row[f"{name}_p{p}"] = pct_from_channel(ch, hit, name, p)\n    return row\n\ndef quick_verdict_rows(beam_rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:\n    """Compact one-row-per-run comparison sheet with alpha5.1 ratio diagnostics.\n\n    This is the first file to open after a Colab run. It includes final class\n    fractions plus the ratio gates that explain why partial_occluder was kept\n    or suppressed.\n    """\n    rows: List[Dict[str, Any]] = []\n    for r in beam_rows:\n        uncertain = float(r.get("uncertain_frac_of_hit_px") or 0.0)\n        solid = float(r.get("solid_surface_frac_of_hit_px") or 0.0)\n        partial = float(r.get("partial_occluder_frac_of_hit_px") or 0.0)\n        partial_cand = float(r.get("partial_candidate_frac_of_hit_px") or 0.0)\n        geom = float(r.get("geom_edge_frac_of_hit_px") or 0.0)\n        split95 = r.get("beam_split_score_p95")\n        coherence50 = r.get("beam_coherence_p50")\n        gap_ratio95 = r.get("beam_mode_gap_ratio_p95")\n        back_ratio95 = r.get("beam_back_ratio_p95")\n        coherent_guard = float(r.get("coherent_guard_candidate_frac_of_hit_px") or 0.0)\n\n        status = "watch"\n        notes = []\n        if uncertain > 0.12:\n            status = "too_uncertain"\n            notes.append("uncertain above 12%")\n        elif partial > 0.08:\n            status = "too_partial"\n            notes.append("partial above 8%; inspect candidate gates")\n        elif solid > 0.75 and partial <= 0.04 and uncertain <= 0.06:\n            status = "promising"\n            notes.append("balanced solid/partial/uncertain")\n        elif solid > 0.60:\n            status = "usable"\n            notes.append("solid recovery working")\n        else:\n            notes.append("mixed/needs visual review")\n\n        if partial_cand > partial * 2.5 and partial_cand > 0.01:\n            notes.append("partial candidates exceed final labels")\n        if coherent_guard > 0.15:\n            notes.append(f"coherent_guard_active={coherent_guard:.3f}")\n        if split95 is not None:\n            notes.append(f"split_p95={float(split95):.3f}")\n        if gap_ratio95 is not None:\n            notes.append(f"gap_ratio_p95={float(gap_ratio95):.3f}")\n        if back_ratio95 is not None:\n            notes.append(f"back_ratio_p95={float(back_ratio95):.3f}")\n        if coherence50 is not None:\n            notes.append(f"coherence_p50={float(coherence50):.3f}")\n\n        rows.append({\n            "run_id": r.get("run_id"),\n            "scene": r.get("scene"),\n            "preset": r.get("preset"),\n            "beam_width": r.get("beam_width"),\n            "coverage": r.get("coverage"),\n            "geom_edge_frac_of_hit_px": geom,\n            "partial_occluder_frac_of_hit_px": partial,\n            "foliage_frac_of_hit_px": r.get("foliage_frac_of_hit_px"),\n            "solid_surface_frac_of_hit_px": solid,\n            "uncertain_frac_of_hit_px": uncertain,\n            "partial_candidate_frac_of_hit_px": partial_cand,\n            "partial_geom_overlap_candidate_frac_of_hit_px": r.get("partial_geom_overlap_candidate_frac_of_hit_px"),\n            "split_candidate_frac_of_hit_px": r.get("split_candidate_frac_of_hit_px"),\n            "front_back_candidate_frac_of_hit_px": r.get("front_back_candidate_frac_of_hit_px"),\n            "gap_ratio_candidate_frac_of_hit_px": r.get("gap_ratio_candidate_frac_of_hit_px"),\n            "back_ratio_candidate_frac_of_hit_px": r.get("back_ratio_candidate_frac_of_hit_px"),\n            "coherent_guard_candidate_frac_of_hit_px": r.get("coherent_guard_candidate_frac_of_hit_px"),\n            "coherence_solid_candidate_frac_of_hit_px": r.get("coherence_solid_candidate_frac_of_hit_px"),\n            "return_mix_score_p95": r.get("return_mix_score_p95"),\n            "return_depth_spread_excess_p95": r.get("return_depth_spread_excess_p95"),\n            "beam_split_score_p95": split95,\n            "beam_mode_gap_ratio_p95": gap_ratio95,\n            "beam_back_ratio_p95": back_ratio95,\n            "beam_coherence_p50": coherence50,\n            "beam_coherence_p95": r.get("beam_coherence_p95"),\n            "recommended_status": status,\n            "notes": "; ".join(notes),\n        })\n    return rows\n\n\ndef flatten_counts(diag: Dict[str, Any], run_id: int, scene: str, preset: str) -> List[Dict[str, Any]]:\n    counts = diag.get("classification_counts") or {}\n    total = sum(int(v) for v in counts.values()) or 1\n    return [\n        {\n            "run_id": run_id,\n            "scene": scene,\n            "preset": preset,\n            "label": k,\n            "count": int(v),\n            "frac_of_all_pixels": float(int(v) / total),\n        }\n        for k, v in sorted(counts.items())\n    ]\n\n\ndef write_rows_csv(path: Path, rows: List[Dict[str, Any]]):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    if not rows:\n        path.write_text("", encoding="utf-8")\n        return\n    keys: List[str] = []\n    for r in rows:\n        for k in r.keys():\n            if k not in keys:\n                keys.append(k)\n    with path.open("w", encoding="utf-8", newline="") as f:\n        writer = csv.DictWriter(f, fieldnames=keys, extrasaction="ignore")\n        writer.writeheader()\n        for r in rows:\n            writer.writerow(r)\n\n\ndef make_top_contact_sheets(contact_paths: List[str], out_path: Path, max_images: int = 8) -> Optional[str]:\n    imgs = []\n    for p in contact_paths[:max_images]:\n        if p and os.path.exists(p):\n            im = Image.open(p).convert("RGB")\n            w, h = im.size\n            scale = min(560 / max(w, 1), 1.0)\n            im = im.resize((int(w * scale), int(h * scale)))\n            label = Path(p).parent.name\n            canvas = Image.new("RGB", (im.width, im.height + 26), "white")\n            canvas.paste(im, (0, 26))\n            draw = ImageDraw.Draw(canvas)\n            draw.text((6, 6), label, fill=(0, 0, 0))\n            imgs.append(canvas)\n    if not imgs:\n        return None\n    cols = 2 if len(imgs) > 1 else 1\n    rows = int(math.ceil(len(imgs) / cols))\n    cell_w = max(im.width for im in imgs)\n    cell_h = max(im.height for im in imgs)\n    sheet = Image.new("RGB", (cols * cell_w, rows * cell_h), "white")\n    for i, im in enumerate(imgs):\n        x = (i % cols) * cell_w\n        y = (i // cols) * cell_h\n        sheet.paste(im, (x, y))\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n    sheet.save(out_path)\n    return str(out_path)\n\n\ndef zip_outputs(out_dir: str | Path, zip_path: Optional[str | Path] = None) -> str:\n    out_dir = Path(out_dir)\n    if zip_path is None:\n        zip_path = out_dir.with_suffix(".zip")\n    zip_path = Path(zip_path)\n    if zip_path.exists():\n        zip_path.unlink()\n    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:\n        for p in sorted(out_dir.rglob("*")):\n            if p.is_file():\n                zf.write(p, arcname=p.relative_to(out_dir.parent))\n    return str(zip_path)\n\n\ndef run_plan(engine_path: str, plan_path: str, out_dir: str = DEFAULT_OUT) -> Dict[str, Any]:\n    llw = load_engine(engine_path)\n    plan = load_plan(plan_path)\n    out = Path(out_dir)\n    if out.exists():\n        shutil.rmtree(out)\n    out.mkdir(parents=True, exist_ok=True)\n\n    all_diag: List[Dict[str, Any]] = []\n    beam_rows: List[Dict[str, Any]] = []\n    class_rows: List[Dict[str, Any]] = []\n    structure_rows: List[Dict[str, Any]] = []\n    material_report_rows: List[Dict[str, Any]] = []\n    material_core_rows: List[Dict[str, Any]] = []\n    material_filled_rows: List[Dict[str, Any]] = []\n    boundary_rows: List[Dict[str, Any]] = []\n    contact_paths: List[str] = []\n    candidate_gate_paths: List[str] = []\n    errors: List[Dict[str, Any]] = []\n\n    for idx, row in plan.iterrows():\n        run_id = _to_int(row.get("run_id"), len(all_diag) + 1) or (len(all_diag) + 1)\n        scene = _to_str(row.get("scene"), "demo") or "demo"\n        preset = _to_str(row.get("preset"), "beam_return_debug") or "beam_return_debug"\n        seed = _to_int(row.get("seed"), 42) or 42\n        run_label = f"run_{run_id:02d}_{_safe_name(scene)}_{_safe_name(preset)}"\n        run_dir = out / run_label\n        run_dir.mkdir(parents=True, exist_ok=True)\n        overrides = row_overrides(row)\n\n        print(f"\\n=== run {run_id}: scene={scene} preset={preset} seed={seed} ===")\n        print("overrides:", overrides)\n        try:\n            prims = scene_from_name(llw, scene)\n            result = llw.run_sensor_preset(\n                prims,\n                preset_name=preset,\n                scene_name=f"{_safe_name(scene)}_{run_id:02d}_{_safe_name(preset)}",\n                out_dir=str(run_dir),\n                seed=seed,\n                **overrides,\n            )\n\n            diag = dict(result.get("diagnostics") or {})\n            diag["run_id"] = run_id\n            diag["input_scene"] = scene\n            diag["input_preset"] = preset\n            diag["contact_sheet"] = result.get("paths", {}).get("contact_sheet")\n            diag["channels_npz"] = result.get("paths", {}).get("channels")\n            diag["diagnostics_json"] = result.get("paths", {}).get("diagnostics")\n            all_diag.append(diag)\n\n            contact = result.get("paths", {}).get("contact_sheet")\n            if contact:\n                contact_paths.append(contact)\n\n            gate_sheet = save_candidate_gate_sheet(\n                result.get("channels", {}),\n                diag,\n                run_dir / f"{_safe_name(scene)}_{run_id:02d}_{_safe_name(preset)}_candidate_gates.png",\n            )\n            if gate_sheet:\n                candidate_gate_paths.append(gate_sheet)\n                diag["candidate_gate_sheet"] = gate_sheet\n\n            beam_rows.append(channel_summary(result, run_id, scene, preset))\n            class_rows.extend(flatten_counts(diag, run_id, scene, preset))\n\n            sd = dict(result.get("structure_density") or {})\n            sd.update({"run_id": run_id, "scene": scene, "preset": preset})\n            structure_rows.append(sd)\n\n            for rec in result.get("material_report") or []:\n                rec = dict(rec); rec.update({"run_id": run_id, "scene": scene, "preset": preset})\n                material_report_rows.append(rec)\n            for rec in result.get("material_core_agreement") or []:\n                rec = dict(rec); rec.update({"run_id": run_id, "scene": scene, "preset": preset})\n                material_core_rows.append(rec)\n            for rec in result.get("material_filled_agreement") or []:\n                rec = dict(rec); rec.update({"run_id": run_id, "scene": scene, "preset": preset})\n                material_filled_rows.append(rec)\n            for rec in result.get("boundary_adjacency") or []:\n                rec = dict(rec); rec.update({"run_id": run_id, "scene": scene, "preset": preset})\n                boundary_rows.append(rec)\n\n        except Exception as exc:\n            err = {\n                "run_id": run_id,\n                "scene": scene,\n                "preset": preset,\n                "error": repr(exc),\n            }\n            errors.append(err)\n            print("ERROR:", repr(exc))\n\n    write_rows_csv(out / "sweep_metrics.csv", all_diag)\n    write_rows_csv(out / "beam_return_summary.csv", beam_rows)\n    write_rows_csv(out / "quick_verdict.csv", quick_verdict_rows(beam_rows))\n    write_rows_csv(out / "gate_breakdown.csv", gate_breakdown_rows(beam_rows))\n    write_rows_csv(out / "class_balance_summary.csv", class_balance_rows(beam_rows))\n    write_rows_csv(out / "classification_counts.csv", class_rows)\n    write_rows_csv(out / "structure_density_report.csv", structure_rows)\n    write_rows_csv(out / "material_channel_report.csv", material_report_rows)\n    write_rows_csv(out / "material_core_agreement.csv", material_core_rows)\n    write_rows_csv(out / "material_filled_agreement.csv", material_filled_rows)\n    write_rows_csv(out / "boundary_adjacency_report.csv", boundary_rows)\n\n    edge_cols = [\n        "preset", "input_scene", "run_id", "edge_threshold_used",\n        "edge_score_p90", "edge_score_p95", "edge_score_p99",\n        "edge_score_raw_p95", "edge_score_geom_p95",\n        "return_mix_score_raw_p95", "return_mix_score_p95",\n        "return_depth_spread_p95", "return_depth_spread_excess_p95",\n        "beam_split_score_p95", "beam_mode_gap_p95",\n        "beam_mode_gap_ratio_p95", "beam_back_ratio_p95",\n        "beam_coherence_p50", "beam_coherence_p95",\n        "partial_candidate_count", "partial_candidate_frac_of_hit_px",\n        "geom_edge_candidate_count", "partial_geom_overlap_candidate_count",\n        "return_coverage_p50", "edge_confidence_max",\n        "beam_profile", "beam_width", "edge_score_mode", "edge_fusion_mode",\n        "edge_anti_min", "edge_anti_max", "adaptive_edge_percentile",\n    ]\n    # Alpha5.1 reporting fix: use beam_rows so derived ratio channel\n    # percentiles/candidate counts are present in the compact edge summary.\n    edge_rows = edge_summary_rows_from_beam_rows(beam_rows, all_diag)\n    write_rows_csv(out / "edge_threshold_diagnostics_summary.csv", edge_rows)\n\n    if errors:\n        write_rows_csv(out / "errors.csv", errors)\n\n    top_path = make_top_contact_sheets(contact_paths, out / "top_contact_sheets.png")\n\n    summary = {\n        "engine": str(engine_path),\n        "plan": str(plan_path),\n        "out_dir": str(out),\n        "runs_requested": int(len(plan)),\n        "runs_completed": int(len(all_diag)),\n        "runs_failed": int(len(errors)),\n        "top_contact_sheets": top_path,\n        "candidate_gate_sheets": candidate_gate_paths,\n        "csv_outputs": [\n            "sweep_metrics.csv",\n            "beam_return_summary.csv",\n            "quick_verdict.csv",\n            "gate_breakdown.csv",\n            "class_balance_summary.csv",\n            "classification_counts.csv",\n            "edge_threshold_diagnostics_summary.csv",\n            "structure_density_report.csv",\n            "material_channel_report.csv",\n            "material_core_agreement.csv",\n            "material_filled_agreement.csv",\n            "boundary_adjacency_report.csv",\n        ],\n    }\n    with (out / "harness_summary.json").open("w", encoding="utf-8") as f:\n        json.dump(summary, f, indent=2)\n\n    print("\\nHarness summary:")\n    print(json.dumps(summary, indent=2))\n    return summary\n\n\ndef main(argv: Optional[List[str]] = None):\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--engine", default=None, help="Engine .py file. If omitted, auto-detects lidar_lenses_wave_v080*.py.")\n    parser.add_argument("--plan", default=None, help="Spreadsheet plan .xlsx. If omitted, auto-detects *test_plan*.xlsx.")\n    parser.add_argument("--out", default=DEFAULT_OUT)\n    parser.add_argument("--zip", dest="zip_path", default=None)\n    args = parser.parse_args(argv)\n\n    engine = args.engine or find_first(["lidar_lenses_wave_v080_alpha5_1.py", "lidar_lenses_wave_v080*.py", "lidar_lenses_wave_v*.py"], "engine .py")\n    plan = args.plan or find_first(["lidar_wave_test_plan_v080_alpha5_1.xlsx", "*test_plan*.xlsx", "*.xlsx"], "test plan .xlsx")\n    print("ENGINE:", engine)\n    print("PLAN:", plan)\n    summary = run_plan(engine, plan, args.out)\n    zip_path = zip_outputs(args.out, args.zip_path)\n    print("\\nZIP:", zip_path)\n    return {"summary": summary, "zip_path": zip_path}\n\n\nif __name__ == "__main__":\n    main()\n'
Path('v080_alpha5_1_upload_harness.py').write_text(HARNESS_CODE, encoding='utf-8')
print('Wrote v080_alpha5_1_upload_harness.py')


In [ ]:
import importlib.util, sys, json
from pathlib import Path

spec = importlib.util.spec_from_file_location("v080_alpha5_1_upload_harness", "v080_alpha5_1_upload_harness.py")
harness = importlib.util.module_from_spec(spec)
sys.modules["v080_alpha5_1_upload_harness"] = harness
spec.loader.exec_module(harness)

OUT_DIR = "v080_alpha5_1_test_outputs"
summary = harness.run_plan(ENGINE_PATH, PLAN_PATH, OUT_DIR)
zip_path = harness.zip_outputs(OUT_DIR, "lidar_wave_test_outputs_v080_alpha5_1.zip")

print(json.dumps(summary, indent=2))
print("ZIP:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Download manually:", zip_path)